# Receiver listening and recording
Interrupt any older running recording cell first. Run the single cell below, then click **Start listening**. The cell returns immediately after showing the controls.
Use **Recording ON/OFF** and **Volume boost** while listening. Files are saved in **recordings**. Close receiver browser audio streams first.


In [1]:
# One cell: install missing packages, display controls, and listen in a background thread.
import importlib.util
import subprocess
import sys

for package in ("requests", "sounddevice", "ipywidgets"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from pathlib import Path
from datetime import datetime
import threading
import time
import math
from concurrent.futures import ThreadPoolExecutor
import queue
import struct
import wave
import requests
import sounddevice as sd
import ipywidgets as widgets
from IPython.display import display

# Stop the previous listener before rerunning this cell.
if "_receiver_ui" in globals() and _receiver_ui["thread"] and _receiver_ui["thread"].is_alive():
    raise RuntimeError("Click Stop listening, wait for 'Stopped', then rerun this cell.")

recordings_folder = Path.cwd() / "recordings"
recordings_folder.mkdir(exist_ok=True)

ip = widgets.Text(value="192.168.1.143", description="Receiver IP:")
start = widgets.Button(description="Start listening", button_style="success")
stop = widgets.Button(description="Stop listening", button_style="warning", disabled=True)
record = widgets.ToggleButton(value=False, description="Recording OFF", icon="circle")
boost = widgets.FloatSlider(value=1, min=0, max=20, step=0.1,
                            description="Volume boost:", readout_format=".1f",
                            continuous_update=True, style={"description_width": "initial"})
buffer_control = widgets.FloatSlider(value=0.75, min=0.25, max=2.0, step=0.05,
    description="Buffer seconds:", readout_format=".2f", style={"description_width": "initial"})
diagnostics = widgets.Label(value="Waiting for audio.")
status = widgets.Label(value=f"Ready. Files: {recordings_folder}")
log = widgets.Output(layout={"max_height": "180px", "overflow": "auto"})
state = {"thread": None, "stop": threading.Event(), "lock": threading.Lock(),
         "record": False, "gain": 1.0, "pcm": bytearray(), "started": None,
         "number": 0, "speaker_error": None}
_receiver_ui = state
chunk_bytes = 16000 * 2 * 15
save_executor = ThreadPoolExecutor(max_workers=1)
save_jobs = []

def save_locked(partial=False):
    if not state["pcm"]:
        return
    state["number"] += 1
    suffix = "_partial" if partial else ""
    path = recordings_folder / (
        f"{state['started']:%Y%m%d_%H%M%S_%f}_{state['number']:05d}{suffix}.wav")
    data = bytes(state["pcm"])
    state["pcm"].clear()
    def write_file():
        try:
            with wave.open(str(path), "wb") as wav:
                wav.setnchannels(1)
                wav.setsampwidth(2)
                wav.setframerate(16000)
                wav.writeframes(data)
            log.append_stdout(f"Saved {path.name}\n")
        except Exception as exc:
            state["stop"].set()
            log.append_stdout(f"Save error: {exc}\n")
    save_jobs[:] = [job for job in save_jobs if not job.done()]
    if len(save_jobs) >= 8:
        state["pcm"].extend(data)
        raise RuntimeError("Disk writes are falling behind; stopping to preserve pending audio.")
    save_jobs.append(save_executor.submit(write_file))

def recording_changed(change):
    try:
        with state["lock"]:
            state["record"] = bool(change["new"])
            if not state["record"]:
                save_locked(partial=True)
        record.description = "Recording ON" if change["new"] else "Recording OFF"
        record.button_style = "danger" if change["new"] else ""
    except Exception as exc:
        state["stop"].set()
        log.append_stdout(f"Recording error: {exc}\n")

def gain_changed(change):
    with state["lock"]:
        state["gain"] = float(change["new"])

record.observe(recording_changed, names="value")
boost.observe(gain_changed, names="value")

def listen(address):
    target_blocks = math.ceil(float(buffer_control.value) * 16000 / 1024)
    audio_queue = queue.Queue(maxsize=max(64, target_blocks * 3))
    rebufferings = 0
    near_clipped = 0
    boosted_clipped = 0
    last_arrival = None
    longest_gap = 0.0
    last_report = 0.0
    speaker_done = threading.Event()
    dropped = 0
    underflows = 0

    def play_speakers():
        nonlocal underflows, rebufferings, boosted_clipped
        try:
            with sd.RawOutputStream(samplerate=16000, channels=1,
                                    dtype="int16", latency=0.25) as speakers:
                buffering = True
                ever_played = False
                while not speaker_done.is_set() and not state["stop"].is_set():
                    if buffering:
                        if audio_queue.qsize() < target_blocks:
                            state["stop"].wait(0.01)
                            continue
                        buffering = False
                    try:
                        block = audio_queue.get_nowait()
                    except queue.Empty:
                        if ever_played:
                            rebufferings += 1
                        buffering = True
                        continue
                    with state["lock"]:
                        gain = state["gain"]
                    values = struct.unpack("<" + "h" * (len(block) // 2), block)
                    boosted_clipped += sum(abs(v * gain) > 32767 for v in values)
                    if gain != 1:
                        block = struct.pack("<" + "h" * len(values),
                            *(max(-32768, min(32767, round(v * gain))) for v in values))
                    underflows += bool(speakers.write(block))
                    ever_played = True
        except Exception as exc:
            state["speaker_error"] = str(exc)
            state["stop"].set()

    player = threading.Thread(target=play_speakers, daemon=True)
    player.start()
    try:
        with requests.Session() as session:
            session.trust_env = False
            with session.get(f"http://{address}/audio", stream=True, timeout=(5, 5)) as response:
                response.raise_for_status()
                if "application/octet-stream" not in response.headers.get("Content-Type", "").lower():
                    raise RuntimeError("Expected receiver 16-bit PCM /audio stream.")
                status.value = "Listening. Toggle Recording ON to save 15-second WAV files."
                pending = b""
                for incoming in response.iter_content(chunk_size=2048):
                    if state["stop"].is_set():
                        break
                    now = time.monotonic()
                    if last_arrival is not None:
                        longest_gap = max(longest_gap, now - last_arrival)
                    last_arrival = now
                    combined = pending + incoming
                    aligned = len(combined) // 2 * 2
                    block, pending = combined[:aligned], combined[aligned:]
                    if not block:
                        continue
                    raw_values = struct.unpack("<" + "h" * (len(block) // 2), block)
                    near_clipped += sum(abs(v) >= 32112 for v in raw_values)
                    with state["lock"]:
                        if state["record"]:
                            offset = 0
                            while offset < len(block):
                                if not state["pcm"]:
                                    state["started"] = datetime.now()
                                count = min(chunk_bytes - len(state["pcm"]), len(block) - offset)
                                state["pcm"].extend(block[offset:offset + count])
                                offset += count
                                if len(state["pcm"]) == chunk_bytes:
                                    save_locked()
                    try:
                        audio_queue.put_nowait(block)
                    except queue.Full:
                        dropped += 1
                    if now - last_report >= 1:
                        diagnostics.value = (
                            f"Queued: {audio_queue.qsize() * 64} ms | Refills: {rebufferings} | "
                            f"Speaker underflows: {underflows} | Monitor drops: {dropped} | "
                            f"Longest delivery gap: {longest_gap * 1000:.0f} ms | "
                            f"Raw near-clipping: {near_clipped} | Boost clipping: {boosted_clipped}")
                        last_report = now
                if not state["stop"].is_set():
                    raise RuntimeError("Receiver disconnected.")
    except Exception as exc:
        if not state["stop"].is_set():
            log.append_stdout(f"Error: {exc}\nClose receiver browser streams and verify the IP.\n")
    finally:
        speaker_done.set()
        player.join(timeout=3)
        if state["speaker_error"]:
            log.append_stdout(f"Speaker error: {state['speaker_error']}\n")
        try:
            with state["lock"]:
                state["record"] = False
                save_locked(partial=True)
        except Exception as exc:
            log.append_stdout(f"Save error: {exc}\n")
        for job in list(save_jobs):
            job.result()
        record.value = False
        buffer_control.disabled = False
        start.disabled = False
        stop.disabled = True
        ip.disabled = False
        status.value = f"Stopped. Monitor underflows: {underflows}; monitor-only dropped blocks: {dropped}."

def start_clicked(_):
    if state["thread"] and state["thread"].is_alive():
        return
    state["stop"].clear()
    state["speaker_error"] = None
    buffer_control.disabled = True
    start.disabled = True
    stop.disabled = False
    ip.disabled = True
    status.value = "Connecting..."
    state["thread"] = threading.Thread(target=listen, args=(ip.value.strip(),), daemon=True)
    state["thread"].start()

def stop_clicked(_):
    state["stop"].set()
    record.value = False
    status.value = "Stopping (up to 5 seconds if the receiver is not responding)..."

start.on_click(start_clicked)
stop.on_click(stop_clicked)
display(widgets.VBox([
    ip, widgets.HBox([start, stop, record]), boost, buffer_control,
    widgets.Label("Buffer applies on Start. More buffering adds listening delay; recordings stay unchanged."),
    widgets.Label("Boost affects speakers only. Recordings preserve original samples."),
    widgets.Label("Close the receiver browser stream first. Recording OFF saves any partial chunk."),
    status, diagnostics, log
]))
